# IMDB Sentiment Analysis: Comparing LLMs and Classical ML
## CSCE 580 Project B - Fall 2025

**Objective:** Compare fine-tuned DistilBERT, base DistilBERT, GPT-2, and classical ML models for sentiment classification on IMDB movie reviews.

**Models:**
1. Fine-tuned DistilBERT
2. Base DistilBERT (no fine-tuning)
3. Base GPT-2
4. Classical ML (Logistic Regression with TF-IDF)

## Setup and Installation

In [1]:
# Install required packages
!pip install torch torchvision torchaudio
!pip install transformers datasets
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install accelerate

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 2.7 MB/s  0:00:03a 0:00:010:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 1.8 MB/s  0:07:34 eta 0:00:010:00:08m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.5 MB/s  0:01:16 eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 3.5 MB/s  0:00:34 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 4.4 MB/s  0:00:37 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 3.3 MB/s  0:00:00m 5.1 MB/s eta 0:00:01
  Attempting uninstall: triton
    Found existing installation: triton 3.4.0
    Uninstalling triton-3.4.0:
      Successfully uninstalled triton-3.4.0
  Attempting uninstall: nvidia-nccl-cu129;38;114m╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [nvidia-nvshmem-cu12]
    Found existing installation: nvidia-nccl-cu12 2.27.3;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np

# Fix matplotlib backend for headless/display issues
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns

import time
from pathlib import Path

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Transformers
from transformers import (
    DistilBertTokenizer, DistilBertForSequenceClassification,
    GPT2Tokenizer, GPT2ForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

/home/droski/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/droski/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-27 15:29:29.555111: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda


## 2. Data Preprocessing [30 points]

Load and preprocess the IMDB dataset with:
- Tokenization for transformer models
- TF-IDF features for classical ML
- Train/test split (stratified)

In [3]:
# Load IMDB dataset
# Download from: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
# Place IMDB Dataset.csv in the data/ directory

data_path = Path('data/IMDB Dataset.csv')
if not data_path.exists():
    print("ERROR: Please download the IMDB dataset and place it in data/IMDB Dataset.csv")
    print("URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
else:
    df = pd.read_csv(data_path)
    print(f"Dataset loaded: {len(df)} reviews")
    print(f"\nDataset info:")
    print(df.head())
    print(f"\nSentiment distribution:")
    print(df['sentiment'].value_counts())

Dataset loaded: 50000 reviews

Dataset info:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
# Convert sentiment labels to binary (positive=1, negative=0)
df['label'] = (df['sentiment'] == 'positive').astype(int)

# Split data: 80% train, 20% test (stratified)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'].values,
    random_state=42
)

# Further split train into train/val: 90% train, 10% val
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.1,
    stratify=train_labels,
    random_state=42
)

print(f"Training set: {len(train_texts)} samples")
print(f"Validation set: {len(val_texts)} samples")
print(f"Test set: {len(test_texts)} samples")

Training set: 36000 samples
Validation set: 4000 samples
Test set: 10000 samples


### 2.1 Prepare Data for Transformer Models

In [5]:
# Custom Dataset class for transformers
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [6]:
# Initialize tokenizers
distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# GPT-2 needs a padding token
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

print("Tokenizers initialized successfully")

Tokenizers initialized successfully


### 2.2 Prepare Data for Classical ML

In [7]:
# Create TF-IDF features for classical ML
print("Creating TF-IDF features...")
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Fit on training data and transform all sets
train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
val_tfidf = tfidf_vectorizer.transform(val_texts)
test_tfidf = tfidf_vectorizer.transform(test_texts)

print(f"TF-IDF shape: {train_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

Creating TF-IDF features...
TF-IDF shape: (36000, 10000)
Vocabulary size: 10000


## 3. Fine-Tuning DistilBERT [50 points]

Fine-tune pre-trained DistilBERT on IMDB training data with monitoring of training/validation loss.

In [8]:
# Hyperparameters
# Configuration optimized for different GPUs:
# - RTX 3090 (24GB): BATCH_SIZE=32, MAX_LENGTH=512 (~20-30min training)
# - Small GPU (4GB): BATCH_SIZE=4, MAX_LENGTH=256 (~2-3hrs training)
# - Current settings work for both

BATCH_SIZE = 32  # Increase to 32 for RTX 3090, use 4 for small GPUs
EPOCHS = 3
LEARNING_RATE = 2e-5
MAX_LENGTH = 512  # Use 256 for small GPUs to save memory

# Create datasets
train_dataset = IMDBDataset(train_texts, train_labels, distilbert_tokenizer, MAX_LENGTH)
val_dataset = IMDBDataset(val_texts, val_labels, distilbert_tokenizer, MAX_LENGTH)
test_dataset = IMDBDataset(test_texts, test_labels, distilbert_tokenizer, MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"DataLoaders created with batch size {BATCH_SIZE}")

DataLoaders created with batch size 16


In [10]:
# Initialize fine-tuned DistilBERT model
finetuned_distilbert = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
).to(device)

# Optimizer and scheduler
optimizer = AdamW(finetuned_distilbert.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(f"Model initialized. Total training steps: {total_steps}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized. Total training steps: 6750


In [11]:
# Training function
def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        logits = outputs.logits
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(dataloader), correct / total

# Evaluation function
def eval_model(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(dataloader), correct / total

In [ ]:
# Train the model
print("Starting fine-tuning DistilBERT...")
training_stats = []
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 50)
    
    train_loss, train_acc = train_epoch(
        finetuned_distilbert, train_loader, optimizer, scheduler, device
    )
    val_loss, val_acc = eval_model(finetuned_distilbert, val_loader, device)
    
    training_stats.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    })
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

finetuned_training_time = time.time() - start_time
print(f"\nFine-tuning completed in {finetuned_training_time:.2f} seconds")

# Save the model
finetuned_distilbert.save_pretrained('models/finetuned_distilbert')
print("Model saved to models/finetuned_distilbert")

Starting fine-tuning DistilBERT...

Epoch 1/3
--------------------------------------------------


## 4. Base Model Comparison [60 points]

Evaluate base DistilBERT and GPT-2 (without fine-tuning) on the test set.

In [ ]:
# Load base DistilBERT (no fine-tuning)
print("Loading base DistilBERT...")
base_distilbert = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
).to(device)

print("Base DistilBERT loaded successfully")

In [ ]:
# Load base GPT-2
print("Loading base GPT-2...")
base_gpt2 = GPT2ForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=2
).to(device)

# Configure GPT-2 for classification
base_gpt2.config.pad_token_id = gpt2_tokenizer.eos_token_id

print("Base GPT-2 loaded successfully")

In [ ]:
# Prepare GPT-2 dataset
test_dataset_gpt2 = IMDBDataset(test_texts, test_labels, gpt2_tokenizer, MAX_LENGTH)
test_loader_gpt2 = DataLoader(test_dataset_gpt2, batch_size=BATCH_SIZE)

print("GPT-2 test dataloader created")

## 5. Classical Machine Learning Model [30 points]

Train Logistic Regression on TF-IDF features.

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
start_time = time.time()

lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(train_tfidf, train_labels)

lr_training_time = time.time() - start_time
print(f"Logistic Regression trained in {lr_training_time:.2f} seconds")

# Validation accuracy
val_preds_lr = lr_model.predict(val_tfidf)
val_acc_lr = accuracy_score(val_labels, val_preds_lr)
print(f"Validation Accuracy: {val_acc_lr:.4f}")

## 6. Model Evaluation

Evaluate all models on the test set and collect predictions.

In [ ]:
# Function to get predictions from transformer models
def get_predictions(model, dataloader, device):
    model.eval()
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    return np.array(predictions), np.array(true_labels)

In [ ]:
# Evaluate all models
print("Evaluating all models on test set...")
print("=" * 50)

# 1. Fine-tuned DistilBERT
print("\n1. Fine-tuned DistilBERT")
start_time = time.time()
preds_finetuned, labels_true = get_predictions(finetuned_distilbert, test_loader, device)
finetuned_inference_time = time.time() - start_time
print(f"Inference time: {finetuned_inference_time:.2f} seconds")

# 2. Base DistilBERT
print("\n2. Base DistilBERT")
start_time = time.time()
preds_base_distilbert, _ = get_predictions(base_distilbert, test_loader, device)
base_distilbert_inference_time = time.time() - start_time
print(f"Inference time: {base_distilbert_inference_time:.2f} seconds")

# 3. Base GPT-2
print("\n3. Base GPT-2")
start_time = time.time()
preds_gpt2, _ = get_predictions(base_gpt2, test_loader_gpt2, device)
gpt2_inference_time = time.time() - start_time
print(f"Inference time: {gpt2_inference_time:.2f} seconds")

# 4. Logistic Regression
print("\n4. Logistic Regression")
start_time = time.time()
preds_lr = lr_model.predict(test_tfidf)
lr_inference_time = time.time() - start_time
print(f"Inference time: {lr_inference_time:.2f} seconds")

print("\n" + "=" * 50)
print("All models evaluated successfully!")

## 7. Analysis and Visualizations

### 7.1 Accuracy and Loss Curves [30 points]

In [ ]:
# Plot training and validation curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

epochs_range = [stat['epoch'] for stat in training_stats]
train_loss = [stat['train_loss'] for stat in training_stats]
val_loss = [stat['val_loss'] for stat in training_stats]
train_acc = [stat['train_acc'] for stat in training_stats]
val_acc = [stat['val_acc'] for stat in training_stats]

# Loss curve
axes[0].plot(epochs_range, train_loss, 'b-o', label='Training Loss')
axes[0].plot(epochs_range, val_loss, 'r-o', label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs_range, train_acc, 'b-o', label='Training Accuracy')
axes[1].plot(epochs_range, val_acc, 'r-o', label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/loss_accuracy_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Analysis:")
print(f"Final Training Loss: {train_loss[-1]:.4f}")
print(f"Final Validation Loss: {val_loss[-1]:.4f}")
print(f"Final Training Accuracy: {train_acc[-1]:.4f}")
print(f"Final Validation Accuracy: {val_acc[-1]:.4f}")

if val_loss[-1] > val_loss[0]:
    print("⚠️ Validation loss increased - possible overfitting")
elif abs(train_acc[-1] - val_acc[-1]) > 0.1:
    print("⚠️ Large gap between train and val accuracy - possible overfitting")
else:
    print("✓ Model is learning well without significant overfitting")

### 7.2 Confusion Matrices [30 points]

In [ ]:
# Generate confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

models_data = [
    ('Fine-tuned DistilBERT', preds_finetuned),
    ('Base DistilBERT', preds_base_distilbert),
    ('Base GPT-2', preds_gpt2),
    ('Logistic Regression', preds_lr)
]

for idx, (model_name, preds) in enumerate(models_data):
    cm = confusion_matrix(labels_true, preds)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    axes[idx].set_title(f'{model_name}\nAccuracy: {accuracy_score(labels_true, preds):.4f}')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')
    
    # Calculate error rates
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    print(f"\n{model_name}:")
    print(f"  True Negatives: {tn}, False Positives: {fp}")
    print(f"  False Negatives: {fn}, True Positives: {tp}")
    print(f"  False Positive Rate: {fpr:.4f}")
    print(f"  False Negative Rate: {fnr:.4f}")

plt.tight_layout()
plt.savefig('results/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

### 7.3 Precision, Recall, and F1-Score [30 points]

In [ ]:
# Calculate metrics for all models
metrics_results = {}

for model_name, preds in models_data:
    accuracy = accuracy_score(labels_true, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_true, preds, average='binary'
    )
    
    metrics_results[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

# Create DataFrame for better visualization
metrics_df = pd.DataFrame(metrics_results).T
print("\n" + "=" * 70)
print("MODEL PERFORMANCE METRICS")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

In [ ]:
# Visualize metrics comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metrics_df.index))
width = 0.2

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, metrics_df[metric], width, label=metric, color=colors[i])

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Performance Metrics Comparison Across Models')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_df.index, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.0])

plt.tight_layout()
plt.savefig('results/metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### 7.4 Performance Comparison Table [30 points]

In [ ]:
# Create comprehensive comparison table
comparison_data = {
    'Model': ['Fine-tuned DistilBERT', 'Base DistilBERT', 'Base GPT-2', 'Logistic Regression'],
    'Accuracy': [metrics_results[m]['Accuracy'] for m in models_data],
    'Precision': [metrics_results[m]['Precision'] for m in models_data],
    'Recall': [metrics_results[m]['Recall'] for m in models_data],
    'F1-Score': [metrics_results[m]['F1-Score'] for m in models_data]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(4)

# Add ranking
comparison_df['Rank'] = comparison_df['F1-Score'].rank(ascending=False).astype(int)
comparison_df = comparison_df.sort_values('Rank')

print("\n" + "=" * 80)
print("COMPREHENSIVE PERFORMANCE COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Save to CSV
comparison_df.to_csv('results/performance_comparison.csv', index=False)
print("\n✓ Saved to results/performance_comparison.csv")

### 7.5 Time Complexity Analysis [30 points]

In [ ]:
# Time complexity comparison
time_data = {
    'Model': ['Fine-tuned DistilBERT', 'Base DistilBERT', 'Base GPT-2', 'Logistic Regression'],
    'Training Time (s)': [finetuned_training_time, 0, 0, lr_training_time],
    'Inference Time (s)': [
        finetuned_inference_time,
        base_distilbert_inference_time,
        gpt2_inference_time,
        lr_inference_time
    ]
}

time_df = pd.DataFrame(time_data)
time_df['Inference Time per Sample (ms)'] = (
    time_df['Inference Time (s)'] / len(test_labels) * 1000
).round(2)

print("\n" + "=" * 80)
print("TIME COMPLEXITY ANALYSIS")
print("=" * 80)
print(time_df.to_string(index=False))
print("=" * 80)

# Visualize time comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Training time
training_times = time_df[time_df['Training Time (s)'] > 0]
axes[0].bar(training_times['Model'], training_times['Training Time (s)'], color='steelblue')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Training Time Comparison')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, alpha=0.3, axis='y')

# Inference time
axes[1].bar(time_df['Model'], time_df['Inference Time per Sample (ms)'], color='coral')
axes[1].set_ylabel('Time (milliseconds per sample)')
axes[1].set_title('Inference Time Comparison')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/time_complexity.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Efficiency Analysis:")
fastest_inference = time_df.loc[time_df['Inference Time per Sample (ms)'].idxmin()]
print(f"Most efficient for inference: {fastest_inference['Model']}")
print(f"  ({fastest_inference['Inference Time per Sample (ms)']:.2f} ms per sample)")

## 8. AI Test Cases [30 points]

Create test cases with varying complexity and evaluate all models using GAICO format.

In [ ]:
# Define test cases with different complexities
test_cases = [
    {
        'id': 'TC001',
        'name': 'Simple positive review',
        'complexity': 'Low',
        'text': 'This movie is great!',
        'expected_sentiment': 1,  # Positive
        'word_count': 4,
        'sentence_count': 1
    },
    {
        'id': 'TC002',
        'name': 'Medium complexity negative review',
        'complexity': 'Medium',
        'text': 'I was really disappointed with this film. The acting was wooden and the plot made no sense. Would not recommend to anyone.',
        'expected_sentiment': 0,  # Negative
        'word_count': 24,
        'sentence_count': 3
    },
    {
        'id': 'TC003',
        'name': 'Complex mixed sentiment review',
        'complexity': 'High',
        'text': 'While the cinematography was absolutely stunning and the lead actor gave a compelling performance, the movie ultimately failed to deliver on its promise. The pacing was slow, the dialogue felt forced, and the ending was predictable. Despite some brilliant moments, I left the theater feeling unsatisfied.',
        'expected_sentiment': 0,  # Overall negative
        'word_count': 53,
        'sentence_count': 4
    },
    {
        'id': 'TC004',
        'name': 'Sarcastic negative review',
        'complexity': 'High',
        'text': 'Oh wow, another masterpiece! Two hours of my life I will never get back. Absolutely brilliant waste of time.',
        'expected_sentiment': 0,  # Negative (sarcasm)
        'word_count': 22,
        'sentence_count': 3
    }
]

print("Test cases created:")
for tc in test_cases:
    print(f"\n{tc['id']}: {tc['name']}")
    print(f"  Complexity: {tc['complexity']}")
    print(f"  Words: {tc['word_count']}, Sentences: {tc['sentence_count']}")
    print(f"  Expected: {'Positive' if tc['expected_sentiment'] == 1 else 'Negative'}")

In [ ]:
# Function to predict sentiment for a single text
def predict_sentiment(text, model, tokenizer, device, max_length=512):
    model.eval()
    
    encoding = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(logits, dim=1).item()
        confidence = probs[0][pred].item()
    
    return pred, confidence

# Function to predict with classical model
def predict_classical(text, model, vectorizer):
    text_tfidf = vectorizer.transform([text])
    pred = model.predict(text_tfidf)[0]
    proba = model.predict_proba(text_tfidf)[0]
    confidence = proba[pred]
    return pred, confidence

In [ ]:
# Evaluate all models on test cases
testcase_results = []

for tc in test_cases:
    text = tc['text']
    expected = tc['expected_sentiment']
    
    # Predict with all models
    pred_ft, conf_ft = predict_sentiment(text, finetuned_distilbert, distilbert_tokenizer, device)
    pred_base, conf_base = predict_sentiment(text, base_distilbert, distilbert_tokenizer, device)
    pred_gpt2, conf_gpt2 = predict_sentiment(text, base_gpt2, gpt2_tokenizer, device)
    pred_lr, conf_lr = predict_classical(text, lr_model, tfidf_vectorizer)
    
    result = {
        'Test Case ID': tc['id'],
        'Name': tc['name'],
        'Complexity': tc['complexity'],
        'Word Count': tc['word_count'],
        'Expected': 'Positive' if expected == 1 else 'Negative',
        'Fine-tuned DistilBERT': f"{'✓' if pred_ft == expected else '✗'} ({conf_ft:.3f})",
        'Base DistilBERT': f"{'✓' if pred_base == expected else '✗'} ({conf_base:.3f})",
        'Base GPT-2': f"{'✓' if pred_gpt2 == expected else '✗'} ({conf_gpt2:.3f})",
        'Logistic Regression': f"{'✓' if pred_lr == expected else '✗'} ({conf_lr:.3f})"
    }
    
    testcase_results.append(result)

# Display results
testcase_df = pd.DataFrame(testcase_results)
print("\n" + "=" * 120)
print("AI TEST CASES EVALUATION (GAICO Format)")
print("=" * 120)
print(testcase_df.to_string(index=False))
print("=" * 120)
print("\n✓ = Correct prediction, ✗ = Incorrect prediction")
print("Numbers in parentheses show confidence scores")

# Save results
testcase_df.to_csv('results/testcase_results.csv', index=False)
print("\n✓ Saved to results/testcase_results.csv")

In [ ]:
# Detailed test case analysis
print("\n" + "=" * 80)
print("DETAILED TEST CASE ANALYSIS")
print("=" * 80)

for idx, tc in enumerate(test_cases):
    print(f"\n{tc['id']}: {tc['name']}")
    print("-" * 80)
    print(f"Text: {tc['text'][:100]}{'...' if len(tc['text']) > 100 else ''}")
    print(f"Complexity: {tc['complexity']} | Words: {tc['word_count']} | Sentences: {tc['sentence_count']}")
    print(f"Expected Sentiment: {'Positive' if tc['expected_sentiment'] == 1 else 'Negative'}")
    print("\nModel Predictions:")
    
    result = testcase_results[idx]
    print(f"  Fine-tuned DistilBERT: {result['Fine-tuned DistilBERT']}")
    print(f"  Base DistilBERT:       {result['Base DistilBERT']}")
    print(f"  Base GPT-2:            {result['Base GPT-2']}")
    print(f"  Logistic Regression:   {result['Logistic Regression']}")

## 9. Analysis Questions [50 points]

Answer the 5 key questions from the project requirements.

### Question 1: What do the accuracy and loss curves tell you about the fine-tuning process?

**Answer:**

The accuracy and loss curves provide insights into how the model learns during fine-tuning:

- **Training Loss**: Should decrease steadily, indicating the model is learning to minimize prediction errors on training data
- **Validation Loss**: Should also decrease. If it starts increasing while training loss decreases, this indicates overfitting
- **Training Accuracy**: Should increase as the model learns patterns in the data
- **Validation Accuracy**: Should track training accuracy. A large gap suggests overfitting

Based on my curves:
- [Analyze specific curves here]
- The model shows [good generalization / signs of overfitting / underfitting]
- The learning process is [stable / unstable] as evidenced by [smooth curves / erratic behavior]

---

### Question 2: How does the fine-tuned DistilBERT model compare to the classical ML model? What advantages or limitations do transformers present over classical algorithms?

**Answer:**

**Performance Comparison:**
- Fine-tuned DistilBERT achieved [X]% accuracy vs Logistic Regression's [Y]% accuracy
- DistilBERT shows [better/worse] precision and recall, particularly on [specific types of reviews]

**Advantages of Transformers:**
1. **Contextual Understanding**: Capture semantic meaning and word relationships through attention mechanisms
2. **Transfer Learning**: Pre-trained on massive datasets, bringing general language understanding
3. **Complex Patterns**: Can model long-range dependencies and nuanced sentiment (e.g., sarcasm)
4. **No Feature Engineering**: Automatically learns relevant features from raw text

**Limitations of Transformers:**
1. **Computational Cost**: Significantly slower training and inference (see time complexity results)
2. **Resource Requirements**: Need GPUs and more memory
3. **Data Efficiency**: May require more data for fine-tuning to reach peak performance
4. **Interpretability**: Harder to understand why specific predictions were made

**When to Use Each:**
- **Transformers**: When accuracy is paramount, data is abundant, and resources are available
- **Classical ML**: When speed, simplicity, and interpretability are priorities, or resources are limited

---

### Question 3: What insights can you draw from the confusion matrix? Are there any patterns in the misclassifications?

**Answer:**

**Key Observations from Confusion Matrices:**

[Analyze your specific results]

**Potential Patterns in Misclassifications:**
1. **False Positives** (Negative labeled as Positive):
   - May occur with sarcastic negative reviews
   - Mixed reviews with positive language but negative overall sentiment
   
2. **False Negatives** (Positive labeled as Negative):
   - Understated positive reviews
   - Reviews with qualified praise ("good but...")

3. **Model-Specific Patterns**:
   - Base models show more balanced errors (less trained)
   - Fine-tuned model may be better at [specific type of review]
   - Classical ML may struggle with [specific linguistic patterns]

**Comparison Across Models:**
- Fine-tuned DistilBERT shows the fewest misclassifications overall
- Base models have higher error rates, especially [FP/FN]
- Logistic Regression performs [similarly/differently] on [specific patterns]

---

### Question 4: Why might the fine-tuned model outperform the base model?

**Answer:**

The fine-tuned DistilBERT outperforms the base model for several reasons:

**1. Domain Adaptation:**
- Base model was pre-trained on general text (Wikipedia, BookCorpus)
- Fine-tuning adapts it specifically to movie review language and sentiment
- Learns domain-specific vocabulary and patterns (e.g., "plot twist", "character development")

**2. Task-Specific Learning:**
- Base model wasn't trained for sentiment classification
- Fine-tuning teaches the model to map text to positive/negative labels
- Updates weights to focus on sentiment-bearing features

**3. Contextual Refinement:**
- Fine-tuning adjusts attention mechanisms for sentiment-relevant words
- Learns to weigh emotional language more heavily
- Better handles negations, intensifiers, and sentiment modifiers

**4. Classification Head Training:**
- The classification layer is trained from scratch during fine-tuning
- Base model's random classification layer makes essentially random predictions
- Fine-tuned layer learns optimal decision boundaries for IMDB data

**Performance Gap:**
- Our results show [X]% improvement from base to fine-tuned model
- This demonstrates the power of transfer learning and domain adaptation

---

### Question 5: Which model would you recommend for deployment in a real-world scenario, and why? Consider both performance and efficiency.

**Answer:**

**Recommendation depends on use case requirements:**

**Scenario 1: High-Volume, Real-Time Application (e.g., social media monitoring)**
- **Recommended: Logistic Regression**
- **Rationale:**
  - Inference time: ~[X] ms per sample vs [Y] ms for DistilBERT
  - Can handle millions of reviews per day on modest hardware
  - Still achieves respectable [Z]% accuracy
  - Easy to deploy, update, and maintain
  - Interpretable: can explain why reviews are classified certain ways

**Scenario 2: Critical Accuracy Application (e.g., professional review analysis)**
- **Recommended: Fine-tuned DistilBERT**
- **Rationale:**
  - Highest accuracy: [X]% vs [Y]% for classical ML
  - Better handles complex language, sarcasm, and nuance
  - Worth the computational cost when accuracy is paramount
  - Can be optimized with model quantization, distillation for deployment

**Scenario 3: Balanced Production System**
- **Recommended: Hybrid Approach**
- **Strategy:**
  1. Use Logistic Regression as primary classifier (fast, cheap)
  2. Route low-confidence predictions to DistilBERT for verification
  3. Achieves 90% of transformer performance at 20% of the cost

**Practical Considerations:**
- **Infrastructure**: GPU availability, latency requirements, budget
- **Scale**: Number of predictions per second needed
- **Accuracy Requirements**: Cost of misclassification
- **Maintenance**: Team expertise, model update frequency

**My Final Recommendation:**
For most real-world scenarios, I would recommend starting with **Logistic Regression** because:
- Achieves [X]% accuracy, often "good enough" for business needs
- [Y]x faster inference enables real-time processing
- Much easier to deploy, monitor, and debug
- Lower operational costs
- Can always upgrade to transformers later if accuracy becomes critical

---

## 10. Summary and Export Results

In [ ]:
# Create results directory structure
import os
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("\n" + "=" * 80)
print("PROJECT SUMMARY")
print("=" * 80)

print("\n📊 Dataset Statistics:")
print(f"  Total Reviews: {len(df)}")
print(f"  Training Set: {len(train_texts)}")
print(f"  Validation Set: {len(val_texts)}")
print(f"  Test Set: {len(test_texts)}")

print("\n🏆 Model Performance Rankings (by F1-Score):")
for idx, row in comparison_df.iterrows():
    print(f"  {row['Rank']}. {row['Model']}: {row['F1-Score']:.4f}")

print("\n⚡ Efficiency Rankings (by Inference Speed):")
time_df_sorted = time_df.sort_values('Inference Time per Sample (ms)')
for idx, row in time_df_sorted.iterrows():
    print(f"  {idx+1}. {row['Model']}: {row['Inference Time per Sample (ms)']:.2f} ms/sample")

print("\n📁 Generated Files:")
print("  - results/loss_accuracy_curves.png")
print("  - results/confusion_matrices.png")
print("  - results/metrics_comparison.png")
print("  - results/time_complexity.png")
print("  - results/performance_comparison.csv")
print("  - results/testcase_results.csv")
print("  - models/finetuned_distilbert/")

print("\n✅ Project Complete!")
print("=" * 80)

## Deliverables Checklist

✅ **Code:**
- [ ] Jupyter Notebook with all implementations
- [ ] Data preprocessing (30 points)
- [ ] Fine-tuned DistilBERT (50 points)
- [ ] Base model evaluations (60 points)
- [ ] Classical ML model (30 points)

✅ **Analysis & Visualizations:**
- [ ] AI test cases (30 points)
- [ ] Accuracy and loss curves (30 points)
- [ ] Confusion matrices (30 points)
- [ ] Precision, recall, F1-score (30 points)
- [ ] Performance comparison table (30 points)
- [ ] Time complexity analysis (30 points)

✅ **Questions:**
- [ ] 5 analysis questions answered (50 points)

✅ **Report:**
- [ ] Final report (PDF/Markdown) with all plots and answers

**Total Points: 400**